<a href="https://colab.research.google.com/github/steamulater/adaptyv-rbx1-binder-design/blob/main/RFdiffussion_RBX1_adaptyv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os, subprocess
from google.colab import drive

# Mount Drive — but stay in /content
drive.mount('/content/drive')

# Define paths using absolute references only (no os.chdir)
WORKDIR = "/content/drive/MyDrive/adaptyv_competition"  # adjust if needed
OUTPUT_DIR = f"{WORKDIR}/rfdiffusion_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify PDB is accessible
pdb_path = f"{WORKDIR}/rbx1_ring_renumbered.pdb"
assert os.path.exists(pdb_path), f"PDB not found at {pdb_path}"
print(f"PDB found: {pdb_path}")
print(f"Output dir: {OUTPUT_DIR}")

Mounted at /content/drive
PDB found: /content/drive/MyDrive/adaptyv_competition/rbx1_ring_renumbered.pdb
Output dir: /content/drive/MyDrive/adaptyv_competition/rfdiffusion_outputs


In [8]:
import subprocess

steps = [
    ("Clone RFdiffusion",   "git clone -q https://github.com/RosettaCommons/RFdiffusion.git /content/RFdiffusion"),
    ("Install se3-transformer-pytorch", "pip -q install git+https://github.com/lucidrains/se3-transformer-pytorch"),
    ("Install RFdiffusion", "pip -q install -e /content/RFdiffusion"),
    ("Install omegaconf",   "pip -q install omegaconf pyrsistent"),
]
for name, cmd in steps:
    r = subprocess.run(cmd, shell=True, cwd="/content", capture_output=True, text=True)
    print(f"{'OK' if r.returncode == 0 else 'FAIL'}: {name}")
    if r.returncode != 0:
        print(r.stderr[-300:])

FAIL: Clone RFdiffusion
fatal: destination path '/content/RFdiffusion' already exists and is not an empty directory.

OK: Install se3-transformer-pytorch
OK: Install RFdiffusion
OK: Install omegaconf


In [5]:
  import subprocess

  steps = [
      ("Install se3-transformer", "pip -q install /content/RFdiffusion/env/SE3Transformer"),
      ("Install RFdiffusion",     "pip -q install -e /content/RFdiffusion"),
  ]
  for name, cmd in steps:
      r = subprocess.run(cmd, shell=True, cwd="/content", capture_output=True, text=True)
      print(f"{'OK' if r.returncode == 0 else 'FAIL'}: {name}")
      if r.returncode != 0:
          print(r.stderr[-400:])

OK: Install se3-transformer
OK: Install RFdiffusion


In [6]:
%%bash
  mkdir -p /content/RFdiffusion/models
  cd /content/RFdiffusion/models
  wget -q https://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc6/Base_ckpt.pt
  wget -q https://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt
  echo "Weights downloaded:"
  ls -lh .

Weights downloaded:
total 462M
-rw-r--r-- 1 root root 462M Mar 30  2023 Complex_base_ckpt.pt


In [14]:
import os, subprocess

os.makedirs("/content/RFdiffusion/models", exist_ok=True)
weights = {
    "Complex_base_ckpt.pt": "https://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt",
    "Base_ckpt.pt":         "https://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc6/Base_ckpt.pt",
}
for fname, url in weights.items():
    out = f"/content/RFdiffusion/models/{fname}"
    # Check if file exists AND is not empty
    if not (os.path.exists(out) and os.path.getsize(out) > 0):
        print(f"Downloading {fname}...")
        subprocess.run(f"wget -q {url} -O {out}", shell=True, cwd="/content")
    print(f"  {fname}: {os.path.getsize(out)/1e6:.0f} MB")

# Once weights confirm (~200 MB each), Cell 4 — run diffusion:

import os, subprocess

WORKDIR   = "/content/drive/MyDrive/adaptyv_competition"
OUTPUT_DIR = f"{WORKDIR}/rfdiffusion_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = (
    "python /content/RFdiffusion/scripts/run_inference.py "
    f"inference.input_pdb={WORKDIR}/rbx1_ring_renumbered.pdb "
    f"inference.output_prefix={OUTPUT_DIR}/rbx1_binder "
    "contigmap.contigs='[A1-77/0 60-100]' "
    "ppi.hotspot_res='[A4,A6,A12,A14,A15,A23,A24,A26,A28,A52,A56,A60,A64,A66]' "
    "inference.num_designs=200 "
    "inference.ckpt_override_path=/content/RFdiffusion/models/Complex_base_ckpt.pt "
    "denoiser.noise_scale_ca=1.0 "
    "denoiser.noise_scale_frame=1.0"
)

print("Running RFdiffusion (200 designs)...")
print(cmd)
result = subprocess.run(cmd, shell=True, cwd="/content",
                        capture_output=False, text=True)
print("Exit code:", result.returncode)

  Complex_base_ckpt.pt: 484 MB
  Base_ckpt.pt: 0 MB
Running RFdiffusion (200 designs)...
python /content/RFdiffusion/scripts/run_inference.py inference.input_pdb=/content/drive/MyDrive/adaptyv_competition/rbx1_ring_renumbered.pdb inference.output_prefix=/content/drive/MyDrive/adaptyv_competition/rfdiffusion_outputs/rbx1_binder contigmap.contigs='[A1-77/0 60-100]' ppi.hotspot_res='[A4,A6,A12,A14,A15,A23,A24,A26,A28,A52,A56,A60,A64,A66]' inference.num_designs=200 inference.ckpt_override_path=/content/RFdiffusion/models/Complex_base_ckpt.pt denoiser.noise_scale_ca=1.0 denoiser.noise_scale_frame=1.0
Exit code: 1


In [10]:
  import os, subprocess

  WORKDIR    = "/content/drive/MyDrive/adaptyv_competition"
  OUTPUT_DIR = f"{WORKDIR}/rfdiffusion_outputs"

  cmd = (
      "python /content/RFdiffusion/scripts/run_inference.py "
      f"inference.input_pdb={WORKDIR}/rbx1_ring_renumbered.pdb "
      f"inference.output_prefix={OUTPUT_DIR}/rbx1_binder "
      "contigmap.contigs='[A1-77/0 60-100]' "
      "ppi.hotspot_res='[A4,A6,A12,A14,A15,A23,A24,A26,A28,A52,A56,A60,A64,A66]' "
      "inference.num_designs=200 "
      "inference.ckpt_override_path=/content/RFdiffusion/models/Complex_base_ckpt.pt "
      "denoiser.noise_scale_ca=1.0 "
      "denoiser.noise_scale_frame=1.0"
  )

  result = subprocess.run(cmd, shell=True, cwd="/content",
                          capture_output=True, text=True)
  # Print last 60 lines of stderr (where the actual error will be)
  print("=== STDERR (last 60 lines) ===")
  print("\n".join(result.stderr.splitlines()[-60:]))
  print("\n=== STDOUT (last 20 lines) ===")
  print("\n".join(result.stdout.splitlines()[-20:]))
  print("\nExit code:", result.returncode)


=== STDERR (last 60 lines) ===
/content/RFdiffusion/scripts/run_inference.py:63: SyntaxWarning: invalid escape sequence '\d'
  m = re.match(".*_(\d+)\.pdb$", e)
Traceback (most recent call last):
  File "/content/RFdiffusion/scripts/run_inference.py", line 22, in <module>
    import hydra
ModuleNotFoundError: No module named 'hydra'

=== STDOUT (last 20 lines) ===


Exit code: 1


In [11]:
  import subprocess

  r = subprocess.run("pip -q install hydra-core", shell=True, cwd="/content",
                     capture_output=True, text=True)
  print("OK" if r.returncode == 0 else r.stderr[-200:])

OK


In [17]:
import os, subprocess

WORKDIR    = "/content/drive/MyDrive/adaptyv_competition"
OUTPUT_DIR = f"{WORKDIR}/rfdiffusion_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = (
    "python /content/RFdiffusion/scripts/run_inference.py "
    f"inference.input_pdb={WORKDIR}/rbx1_ring_renumbered.pdb "
    f"inference.output_prefix={OUTPUT_DIR}/rbx1_binder "
    "contigmap.contigs='[A1-77/0 60-100]' "
    "ppi.hotspot_res='[A4,A6,A12,A14,A15,A23,A24,A26,A28,A52,A56,A60,A64,A66]' "
    "inference.num_designs=200 "
    "inference.ckpt_override_path=/content/RFdiffusion/models/Complex_base_ckpt.pt "
    "denoiser.noise_scale_ca=1.0 "
    "denoiser.noise_scale_frame=1.0"
)

result = subprocess.run(cmd, shell=True, cwd="/content",
                        capture_output=True, text=True)
print("=== STDERR (last 60 lines) ===")
print("\n".join(result.stderr.splitlines()[-60:]))
print("\n=== STDOUT (last 20 lines) ===")
print("\n".join(result.stdout.splitlines()[-20:]))
print("\nExit code:", result.returncode)

=== STDERR (last 60 lines) ===
/content/RFdiffusion/scripts/run_inference.py:63: SyntaxWarning: invalid escape sequence '\d'
  m = re.match(".*_(\d+)\.pdb$", e)
/content/RFdiffusion/rfdiffusion/util.py:253: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:63.)
  Z = torch.cross(Xn, Yn)
DGL backend not selected or invalid.  Assuming PyTorch for now.
Traceback (most recent call last):
  File "/content/RFdiffusion/scripts/run_inference.py", line 25, in <module>
    from rfdiffusion.inference import utils as iu
  File "/content/RFdiffusion/rfdiffusion/inference/utils.py", line 6, in <module>
    from rfdiffusion.diffusion import get_beta_schedule
  File "/content/RFdiffusion/rfdiffusion/diffusion.py", line 12, in <module>
    from rfdif

In [22]:
import subprocess

r = subprocess.run("pip -q install dgl", shell=True, cwd="/content",
                   capture_output=True, text=True)
print("OK" if r.returncode == 0 else r.stderr[-200:])

OK


In [16]:
  import subprocess, torch

  # Detect current torch and CUDA versions
  torch_ver = torch.__version__.split("+")[0]          # e.g. "2.4.0"
  cuda_ver  = torch.version.cuda.replace(".", "")      # e.g. "124"
  print(f"Torch: {torch_ver}, CUDA: {cuda_ver}")

  steps = [
      f"pip -q install dgl -f https://data.dgl.ai/wheels/torch-{torch_ver}/cu{cuda_ver}/repo.html",
      f"pip -q install torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html",
  ]
  for cmd in steps:
      print(f"Running: {cmd.split()[3]}")
      r = subprocess.run(cmd, shell=True, cwd="/content", capture_output=True, text=True)
      print("OK" if r.returncode == 0 else f"FAIL:\n{r.stderr[-300:]}")


Torch: 2.10.0, CUDA: 128
Running: dgl
OK
Running: torch_scatter
OK


In [20]:
import os, subprocess

WORKDIR    = "/content/drive/MyDrive/adaptyv_competition"
OUTPUT_DIR = f"{WORKDIR}/rfdiffusion_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = (
    "python /content/RFdiffusion/scripts/run_inference.py "
    f"inference.input_pdb={WORKDIR}/rbx1_ring_renumbered.pdb "
    f"inference.output_prefix={OUTPUT_DIR}/rbx1_binder "
    "contigmap.contigs='[A1-77/0 60-100]' "
    "ppi.hotspot_res='[A4,A6,A12,A14,A15,A23,A24,A26,A28,A52,A56,A60,A64,A66]' "
    "inference.num_designs=200 "
    "inference.ckpt_override_path=/content/RFdiffusion/models/Complex_base_ckpt.pt "
    "denoiser.noise_scale_ca=1.0 "
    "denoiser.noise_scale_frame=1.0"
)

result = subprocess.run(cmd, shell=True, cwd="/content",
                        capture_output=True, text=True)
print("=== STDERR (last 60 lines) ===")
print("\n".join(result.stderr.splitlines()[-60:]))
print("\n=== STDOUT (last 20 lines) ===")
print("\n".join(result.stdout.splitlines()[-20:]))
print("\nExit code:", result.returncode)

=== STDERR (last 60 lines) ===
/content/RFdiffusion/scripts/run_inference.py:63: SyntaxWarning: invalid escape sequence '\d'
  m = re.match(".*_(\d+)\.pdb$", e)
/content/RFdiffusion/rfdiffusion/util.py:253: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:63.)
  Z = torch.cross(Xn, Yn)
Traceback (most recent call last):
  File "/content/RFdiffusion/scripts/run_inference.py", line 25, in <module>
    from rfdiffusion.inference import utils as iu
  File "/content/RFdiffusion/rfdiffusion/inference/utils.py", line 6, in <module>
    from rfdiffusion.diffusion import get_beta_schedule
  File "/content/RFdiffusion/rfdiffusion/diffusion.py", line 12, in <module>
    from rfdiffusion.util_module import ComputeAllAtomCoords
  File "/content/

In [19]:
  import subprocess

  r = subprocess.run("pip -q install torchdata==0.7.1", shell=True, cwd="/content",
                     capture_output=True, text=True)
  print("OK" if r.returncode == 0 else r.stderr[-300:])

OK


In [23]:
import subprocess, os

# Remove broken RFdiffusion install
subprocess.run("pip uninstall -y dgl rfdiffusion", shell=True, cwd="/content",
                 capture_output=True)

steps = [
    ("Clone ColabDesign",   "git clone -q https://github.com/sokrypton/ColabDesign.git /content/ColabDesign"),
    ("Install ColabDesign", "pip -q install /content/ColabDesign"),
    ("Clone RFdiffusion",   "git clone -q https://github.com/RosettaCommons/RFdiffusion.git /content/RFdiffusion"),
    ("Install SE3Transformer", "pip -q install /content/RFdiffusion/env/SE3Transformer"),
    ("Install hydra",       "pip -q install hydra-core omegaconf"),
]
for name, cmd in steps:
    r = subprocess.run(cmd, shell=True, cwd="/content", capture_output=True, text=True)
    print(f"{'OK' if r.returncode == 0 else 'FAIL'}: {name}")
    if r.returncode != 0:
        print(r.stderr[-200:])

# Then replace Cell 4 with the ColabDesign API:

import sys, os
sys.path.insert(0, "/content/ColabDesign")
sys.path.insert(0, "/content/RFdiffusion")

from colabdesign.rf.diffusion import mk_diffusion_model

WORKDIR    = "/content/drive/MyDrive/adaptyv_competition"
OUTPUT_DIR = f"{WORKDIR}/rfdiffusion_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

model = mk_diffusion_model(
    weights="/content/RFdiffusion/models/Complex_base_ckpt.pt"
)

model.design(
    pdb          = f"{WORKDIR}/rbx1_ring_renumbered.pdb",
    contigs      = ["A1-77/0 60-100"],
    hotspot_res  = ["A4","A6","A12","A14","A15","A23","A24","A26","A28",
                    "A52","A56","A60","A64","A66"],
    num_designs  = 200,
    output_prefix= f"{OUTPUT_DIR}/rbx1_binder",
)
print("Done")

OK: Clone ColabDesign
OK: Install ColabDesign
FAIL: Clone RFdiffusion
fatal: destination path '/content/RFdiffusion' already exists and is not an empty directory.

OK: Install SE3Transformer
OK: Install hydra


ModuleNotFoundError: No module named 'colabdesign.rf.diffusion'

In [27]:
  import os

  graphbolt_init = "/usr/local/lib/python3.12/dist-packages/dgl/graphbolt/__init__.py"
  with open(graphbolt_init, "r") as f:
      content = f.read()

  patched = content.replace(
      "raise FileNotFoundError(",
      "return  # raise FileNotFoundError("
  )
  with open(graphbolt_init, "w") as f:
      f.write(patched)

  import importlib, sys
  for mod in list(sys.modules.keys()):
      if "dgl" in mod:
          del sys.modules[mod]
  import dgl
  print("DGL OK:", dgl.__version__)

FileNotFoundError: [Errno 2] No such file or directory: '/usr/local/lib/python3.12/dist-packages/dgl/graphbolt/__init__.py'

In [28]:
  import subprocess
  r = subprocess.run("find /usr -name 'graphbolt' -type d 2>/dev/null",
                     shell=True, capture_output=True, text=True)
  print(r.stdout)

  import dgl
  print("DGL path:", dgl.__file__)

ModuleNotFoundError: No module named 'dgl'

In [29]:
  import subprocess, sys
  import torch
  print("Python:", sys.version)
  print("Torch:", torch.__version__)
  print("CUDA:", torch.version.cuda)

  # Find available DGL wheels for this torch version
  r = subprocess.run(
      f"pip index versions dgl 2>/dev/null || pip install dgl== 2>&1 | grep 'from versions'",
      shell=True, capture_output=True, text=True)
  print(r.stdout[-500:] or r.stderr[-500:])


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: 12.8
dgl (2.1.0)
Available versions: 2.1.0, 0.1.3, 0.1.2, 0.1.0



In [31]:
import subprocess

# Uninstall any broken DGL first
subprocess.run("pip uninstall -y dgl", shell=True, capture_output=True)

# Install from DGL's own CUDA 12.1 index
r = subprocess.run(
    "pip install dgl -f https://data.dgl.ai/wheels/cu121/repo.html",
    shell=True, cwd="/content", capture_output=True, text=True
)
print("OK" if r.returncode == 0 else r.stderr[-400:])

# If that works, find and patch the graphbolt file before importing:

import glob, os

# Find graphbolt __init__.py
paths = glob.glob("/usr/local/lib/python*/dist-packages/dgl/graphbolt/__init__.py")
print("Found:", paths)

if paths:
    with open(paths[0], "r") as f:
        content = f.read()
    patched = content.replace(
        "raise FileNotFoundError(",
        "return  # raise FileNotFoundError("
    )
    with open(paths[0], "w") as f:
        f.write(patched)
    print("Patched")

# Verify
import dgl
print("DGL OK:", dgl.__version__)


OK
Found: ['/usr/local/lib/python3.12/dist-packages/dgl/graphbolt/__init__.py']
Patched


IndentationError: unexpected indent (__init__.py, line 46)

In [32]:
  import glob

  paths = glob.glob("/usr/local/lib/python*/dist-packages/dgl/graphbolt/__init__.py")
  path = paths[0]

  with open(path, "r") as f:
      lines = f.readlines()

  # Write back, skipping lines inside the raise FileNotFoundError block
  out = []
  skip = False
  for line in lines:
      if "raise FileNotFoundError(" in line:
          out.append(line.replace("raise FileNotFoundError(", "pass  # "))
          skip = True
          continue
      if skip:
          # Skip continuation lines of the raise block until closing paren
          if ")" in line:
              skip = False
          continue
      out.append(line)

  with open(path, "w") as f:
      f.writelines(out)

  print("Patched. Verifying...")
  import importlib, sys
  for mod in list(sys.modules.keys()):
      if "dgl" in mod:
          del sys.modules[mod]

  import dgl
  print("DGL OK:", dgl.__version__)

Patched. Verifying...
DGL OK: 2.1.0+cu121


In [33]:
  import os, subprocess

  WORKDIR    = "/content/drive/MyDrive/adaptyv_competition"
  OUTPUT_DIR = f"{WORKDIR}/rfdiffusion_outputs"
  os.makedirs(OUTPUT_DIR, exist_ok=True)

  cmd = (
      "python /content/RFdiffusion/scripts/run_inference.py "
      f"inference.input_pdb={WORKDIR}/rbx1_ring_renumbered.pdb "
      f"inference.output_prefix={OUTPUT_DIR}/rbx1_binder "
      "contigmap.contigs='[A1-77/0 60-100]' "
      "ppi.hotspot_res='[A4,A6,A12,A14,A15,A23,A24,A26,A28,A52,A56,A60,A64,A66]' "
      "inference.num_designs=200 "
      "inference.ckpt_override_path=/content/RFdiffusion/models/Complex_base_ckpt.pt "
      "denoiser.noise_scale_ca=1.0 "
      "denoiser.noise_scale_frame=1.0"
  )

  result = subprocess.run(cmd, shell=True, cwd="/content",
                          capture_output=True, text=True)
  print("=== STDERR (last 60 lines) ===")
  print("\n".join(result.stderr.splitlines()[-60:]))
  print("\n=== STDOUT (last 20 lines) ===")
  print("\n".join(result.stdout.splitlines()[-20:]))
  print("\nExit code:", result.returncode)


=== STDERR (last 60 lines) ===
/content/RFdiffusion/scripts/run_inference.py:63: SyntaxWarning: invalid escape sequence '\d'
  m = re.match(".*_(\d+)\.pdb$", e)
Traceback (most recent call last):
  File "/content/RFdiffusion/scripts/run_inference.py", line 24, in <module>
    from rfdiffusion.util import writepdb_multi, writepdb
ModuleNotFoundError: No module named 'rfdiffusion'

=== STDOUT (last 20 lines) ===


Exit code: 1
